![DataKern Watermark](https://img.shields.io/badge/DATAKERN-Spark%20Practice%20Lab-F26522?style=for-the-badge)

# Lab 1 · Read: get the data in correctly

> ## 🎯 The idea of this lab
> This lab is **not** a test of whether you can invent Spark code. Its goal is to build **developer memory**: the way a working data engineer handles Spark.
> *Look at the data → write a transformation → run it → check the result → fix what is wrong → move on.*
> You repeat that loop until the core moves come from your **fingers**, not from a search box.
>
> Almost all the code is given. You **retype it and run it**, and after every run you **write down what you observed**. Typing builds the muscle. Observing turns the muscle into understanding.

**Time:** about 60 to 75 minutes · **Needs:** the Olist CSV files, Java 17+, `pip install -r requirements.txt`

### The scenario
You have just joined the Olist data team. Your lead sends you tickets, one at a time. This notebook has four. Each is about the first job of every pipeline: **reading the data correctly**. Most silent data loss starts here: a wrong type, a split row, a header counted as data. Nothing crashes and the numbers are simply wrong.

### How a ticket works
| Symbol | What you do |
|---|---|
| ⌨️ **Type this** | Retype the code shown into the empty cell and run it. Do not paste. |
| 📝 **Observe** | Look at the output and write what you saw in your own words, right under it. |
| ✍️ **Now you** | The same pattern on new input. Write it yourself, with the typed code above as your model. |
| ✅ **Check** | Run it. A green tick means your result matches the known answer. |

### Three habits to start building today
1. **Type it, don't paste it.** Your fingers learn what your eyes only skim.
2. **Look, do, check.** After every transformation run the same three lines: `df.printSchema()`, `df.show(5)`, `df.count()`.
3. **Write down what you saw.** A number you did not write down is a number you did not notice.

> **Run the cells top to bottom.** When you finish: *Restart* the kernel and *Run All*. No cell should show an error. Use the exact variable names given, because the checks look for them.

---
## Setup: start Spark and point to the files
Run these two cells (nothing to type).

In [1]:
from pyspark.sql import functions as F
from lab_utils import get_spark, olist, check, checks_summary, root_cause, JobCounter

spark = get_spark("Lab 1 · Read")

:: loading settings :: url = jar:file:/private/tmp/claude-501/-Users-ramadugu-Documents-vscode-git-datakern/e760cae3-e247-4ce2-9112-a173b46dc9b9/scratchpad/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ramadugu/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ramadugu/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-56a6a37b-182f-4cb3-8641-75c53a9be96f;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0

	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 213ms :: artifacts dl 6ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from c

⚡ Spark 4.2.0 is running with 16 cores (Delta Lake is on)
🌐 Spark UI ➔ http://127.0.0.1:4040   (open it in your browser and keep the tab open)


In [2]:
ORDERS    = olist("olist_orders_dataset.csv")
CUSTOMERS = olist("olist_customers_dataset.csv")
SELLERS   = olist("olist_sellers_dataset.csv")
REVIEWS   = olist("olist_order_reviews_dataset.csv")

> **First run only:** Spark downloads the Delta Lake jars (1 to 2 minutes, needs internet) and prints red text. That is normal.
> **Open the Spark UI link** printed above in a browser tab and keep it open for the whole lab.
> Write the Spark version and the number of cores at the top of your `OBSERVATION_REPORT.md`.

---
## 🎫 Ticket 1 · "Load the orders and tell me what you see"

> **Lead:** *"Load orders.csv. How many orders do we have, and what does one row look like?"*

**Issue:** a CSV is just text. Spark cannot know that line 1 holds column names or what type each column is.
**See:** what you get with no options, then with the one option that fixes the names.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
raw = spark.read.csv(ORDERS)
raw.printSchema()
raw.show(3, truncate=28)
raw.count()
```

In [3]:
raw = spark.read.csv(ORDERS)
raw.printSchema()
raw.show(3, truncate=28)
raw.count()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)

+----------------------------+----------------------------+------------+------------------------+-------------------+----------------------------+----------------------------+----------------------------+
|                         _c0|                         _c1|         _c2|                     _c3|                _c4|                         _c5|                         _c6|                         _c7|
+----------------------------+----------------------------+------------+------------------------+-------------------+----------------------------+----------------------------+----------------------------+
|                    order_id|                 customer_id|order_status|order_purchase_timestamp|  

99442

📝 **Observe.** What are the columns called? Where did the header line go? What type is every column? What does `count()` say, and is that the number of orders?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
orders = spark.read.csv(ORDERS, header=True)
orders.printSchema()
orders.show(3, truncate=28)
orders.count()
```

In [4]:
orders = spark.read.csv(ORDERS, header=True)
orders.printSchema()
orders.show(3, truncate=28)
orders.count()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

+----------------------------+----------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|                    order_id|                 customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+----------------------------+----------------------------+------------+------------------------+-------------------+----------------------------+--------------------

99441

📝 **Observe.** What changed compared with the first read: the column names, the row count, the types? Which problem is **still there**? (Look at the types, and think about `order_purchase_timestamp`.)

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

**Break it on purpose.** Every developer mistypes a column name. Reading the error well is a skill, so let us cause one. The `try/except` is only there so *Run All* keeps going; in real life you would simply see a red box.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
try:
    orders.select("order_idd")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:330])
```

In [5]:
try:
    orders.select("order_idd")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:330])

{"ts": "2026-11-03 16:01:38.038", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_idd` cannot be resolved. Did you mean one of the following? [`order_id`, `customer_id`, `order_status`, `order_approved_at`, `order_purchase_timestamp`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o61.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_idd` cannot be resolved. Did you mean one of the following? [`order_id`, `customer_id`, `order_status`, `order_approved_at`, `order_purchase_timestamp`]. SQLSTATE: 42703;\n'Project ['order_idd]\n+- Relation [order_id#

AnalysisException
[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `order_idd` cannot be resolved. Did you mean one of the following? [`order_id`, `customer_id`, `order_status`, `order_approved_at`, `order_purchase_timestamp`]. SQLSTATE: 42703;
'Project ['order_idd]
+- Relation [order_id#88,customer_id#89,or


📝 **Observe.** Which part of the message names the problem? What did Spark **suggest**? If it had not suggested anything, what is the quickest way to see the real column names?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Load `customers.csv` (the path is in `CUSTOMERS`) with its header into a variable called **`customers`**. Then do the three-line habit on it: `printSchema()`, `show(3)`, `count()`.

In [6]:
customers = spark.read.csv(CUSTOMERS, header=True)
customers.printSchema()
customers.show(3)
customers.count()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 3 

99441

📝 **Observe.** Did it work first time? If you hit an error, what was it and how did you fix it? (Add it to the bug list in your report.)

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [7]:
check("orders: column names come from the header", lambda: orders.columns[0] == "order_id",
      "use the option that treats line 1 as column names")
check("orders: 99,441 rows (the header is not an order)", lambda: orders.count() == 99_441)
check("customers: loaded with header, 99,441 rows",
      lambda: customers.columns[0] == "customer_id" and customers.count() == 99_441,
      "variable must be called customers, read with header=True")

✅ orders: column names come from the header
✅ orders: 99,441 rows (the header is not an order)


✅ customers: loaded with header, 99,441 rows


---
## 🎫 Ticket 2 · "The date columns are strings. Give me real types. Don't lose anything."

> **Lead:** *"Everything Spark gave us is a string. I need proper types. And last time someone's zip codes got mangled, so be careful."*

**Issue:** `header=True` fixed the names, but every column is still a string. Spark has two ways to get types: `inferSchema=True` (it reads your data to guess) or an explicit `schema` (you declare it).
**See:** what each way costs, and what the guess does to a zip code.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
CUSTOMER_DDL = ("customer_id STRING, customer_unique_id STRING, customer_zip_code_prefix STRING, "
                "customer_city STRING, customer_state STRING")

with JobCounter(spark) as j_infer:
    customers_inferred = spark.read.csv(CUSTOMERS, header=True, inferSchema=True)

with JobCounter(spark) as j_explicit:
    customers_explicit = spark.read.csv(CUSTOMERS, header=True, schema=CUSTOMER_DDL)

print("jobs launched by inferSchema:     ", j_infer.jobs)
print("jobs launched by an explicit schema:", j_explicit.jobs)
customers_inferred.printSchema()
```

In [8]:
CUSTOMER_DDL = ("customer_id STRING, customer_unique_id STRING, customer_zip_code_prefix STRING, "
                "customer_city STRING, customer_state STRING")

with JobCounter(spark) as j_infer:
    customers_inferred = spark.read.csv(CUSTOMERS, header=True, inferSchema=True)

with JobCounter(spark) as j_explicit:
    customers_explicit = spark.read.csv(CUSTOMERS, header=True, schema=CUSTOMER_DDL)

print("jobs launched by inferSchema:     ", j_infer.jobs)
print("jobs launched by an explicit schema:", j_explicit.jobs)
customers_inferred.printSchema()

jobs launched by inferSchema:      2
jobs launched by an explicit schema: 0
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



📝 **Observe.** How many jobs did each read launch **on the read line itself**? What do you think Spark was doing in those jobs? What type did `inferSchema` pick for `customer_zip_code_prefix`?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this (a chained query: each line adds one step; the brackets let it span lines):

```python
side_by_side = (
    customers_explicit.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_declared"))
    .join(customers_inferred.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_guessed")),
          "customer_id")
)
side_by_side.filter(F.col("zip_declared").startswith("0")).show(5)
side_by_side.filter(F.col("zip_declared").startswith("0")).count()
```

In [9]:
side_by_side = (
    customers_explicit.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_declared"))
    .join(customers_inferred.select("customer_id", F.col("customer_zip_code_prefix").alias("zip_guessed")),
          "customer_id")
)
side_by_side.filter(F.col("zip_declared").startswith("0")).show(5)
side_by_side.filter(F.col("zip_declared").startswith("0")).count()

+--------------------+------------+-----------+
|         customer_id|zip_declared|zip_guessed|
+--------------------+------------+-----------+
|18955e83d337fd6b2...|       09790|       9790|
|4e7b3e00288586ebd...|       01151|       1151|
|b2b6027bc5c5109e5...|       08775|       8775|
|fd826e7cf63160e53...|       04534|       4534|
|eabebad39a88bb6f5...|       05704|       5704|
+--------------------+------------+-----------+
only showing top 5 rows


23995

📝 **Observe.** Compare `zip_declared` and `zip_guessed`. What happened to the leading zero? How many customers does this affect? Nothing crashed: why is that **more** dangerous than an error?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Do the same for sellers. First look at the columns with `spark.read.csv(SELLERS, header=True).columns`. Then write a DDL string called **`SELLER_DDL`** (the zip code is an *identifier*: choose its type with Ticket 2 in mind), read the file with it into **`sellers`**, and do the three-line habit.

In [10]:
spark.read.csv(SELLERS, header=True).columns
SELLER_DDL = ("seller_id STRING, seller_zip_code_prefix STRING, seller_city STRING, seller_state STRING")
sellers = spark.read.csv(SELLERS, header=True, schema=SELLER_DDL)
sellers.printSchema()
sellers.show(3)
sellers.count()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



+--------------------+----------------------+--------------+------------+
|           seller_id|seller_zip_code_prefix|   seller_city|seller_state|
+--------------------+----------------------+--------------+------------+
|3442f8959a84dea7e...|                 13023|      campinas|          SP|
|d1b65fc7debc3361e...|                 13844|    mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|rio de janeiro|          RJ|
+--------------------+----------------------+--------------+------------+
only showing top 3 rows


3095

📝 **Observe.** Did it work first time? Which type did you give the zip code and why? If you hit an error, what was it and how did you fix it?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [11]:
check("inferSchema launched jobs, an explicit schema launched none",
      lambda: j_infer.jobs > 0 and j_explicit.jobs == 0)
check("23,995 customers have a zip code starting with 0",
      lambda: side_by_side.filter(F.col("zip_declared").startswith("0")).count() == 23_995)
check("sellers: zip code is a string", lambda: dict(sellers.dtypes)["seller_zip_code_prefix"] == "string",
      "an integer drops the leading zero; a string keeps every character")
check("sellers: every zip code still has 5 characters",
      lambda: sellers.filter(F.length("seller_zip_code_prefix") != 5).count() == 0)
check("sellers: 3,095 rows", lambda: sellers.count() == 3_095)

✅ inferSchema launched jobs, an explicit schema launched none
✅ 23,995 customers have a zip code starting with 0
✅ sellers: zip code is a string


✅ sellers: every zip code still has 5 characters


✅ sellers: 3,095 rows


---
## 🎫 Ticket 3 · "How many reviews do we have?"

> **Lead:** *"Quick one. Load the reviews and give me the count."*

**Issue:** customers type free text into review comments, including **line breaks** and **quote marks**. A reader that works line by line sees one review as several rows.
**See:** the count from a plain read, the rows that look wrong, and the read that fixes them.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
REVIEW_DDL = ("review_id STRING, order_id STRING, review_score INT, review_comment_title STRING, "
              "review_comment_message STRING, review_creation_date TIMESTAMP, review_answer_timestamp TIMESTAMP")

reviews_plain = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL)
reviews_plain.count()
```

In [12]:
REVIEW_DDL = ("review_id STRING, order_id STRING, review_score INT, review_comment_title STRING, "
              "review_comment_message STRING, review_creation_date TIMESTAMP, review_answer_timestamp TIMESTAMP")

reviews_plain = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL)
reviews_plain.count()

104162

📝 **Observe.** Does this number look plausible to you? Write what you would have to check to know whether it is **right**.

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
(reviews_plain.filter(F.col("review_score").isNull())
              .select("review_id", "order_id", "review_score")
              .show(5, truncate=48))
```

In [13]:
(reviews_plain.filter(F.col("review_score").isNull())
              .select("review_id", "order_id", "review_score")
              .show(5, truncate=48))

+------------------------------------------------+-------------------+------------+
|                                       review_id|           order_id|review_score|
+------------------------------------------------+-------------------+------------+
|        ,2018-02-16 00:00:00,2018-02-20 10:52:22|               NULL|        NULL|
|A entrega foi efetuada muito antes do prazo d...|               NULL|        NULL|
|O produto já começou a ser usado e até o pres...|               NULL|        NULL|
|                                 sem problemas."|2018-03-23 00:00:00|        NULL|
|Faltou Controle Remoto para Ar Condicionado C...|2018-01-28 00:00:00|        NULL|
+------------------------------------------------+-------------------+------------+
only showing top 5 rows


📝 **Observe.** These rows have no review score, so they are not reviews. What is sitting in the `review_id` column instead? In your own words: how did one review turn into two or more rows?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

⌨️ **Type this.** Type this. `multiLine` lets a quoted field span lines, `quote` wraps a field, `escape` marks a quote *inside* a field:

```python
reviews = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL,
                         multiLine=True, quote='"', escape='"')
reviews.count()
```

In [14]:
reviews = spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL,
                         multiLine=True, quote='"', escape='"')
reviews.count()

99224

📝 **Observe.** What is the true number of reviews? By how much was the plain read wrong, and in which direction? Why would a pipeline built on the plain read still "work"?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Using `reviews`, answer two questions from the lead and store the numbers in variables: **`one_star`** = how many reviews have a score of 1, and **`with_comment`** = how many have a comment message that is not null. Print both.

In [15]:
one_star = reviews.filter(F.col("review_score") == 1).count()
with_comment = reviews.filter(F.col("review_comment_message").isNotNull()).count()
print(one_star, with_comment)

11424 40977


📝 **Observe.** Did it work first time? Which two methods did you combine to answer each question (for example `filter` then `count`)? If you hit an error, what was it?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [16]:
check("plain read: 104,162 rows (wrong!)", lambda: reviews_plain.count() == 104_162)
check("multiLine read: 99,224 reviews", lambda: reviews.count() == 99_224,
      "multiLine, quote and escape all need to be set")
check("one_star = 11,424", lambda: one_star == 11_424)
check("with_comment = 40,977", lambda: with_comment == 40_977)

✅ plain read: 104,162 rows (wrong!)
✅ multiLine read: 99,224 reviews
✅ one_star = 11,424
✅ with_comment = 40,977


---
## 🎫 Ticket 4 · "What happens when a row does not fit the schema?"

> **Lead:** *"The reviews had junk rows. What would Spark have done with them if we had declared `review_score` as an integer and trusted it?"*

**Issue:** when a row breaks your schema, Spark must do something. You choose with `mode`:

| Mode | Spark's behaviour on a row that does not fit |
|---|---|
| `PERMISSIVE` (default) | keeps the row, puts `null` where a value does not fit |
| `DROPMALFORMED` | throws the row away |
| `FAILFAST` | stops the whole read with an error |

**See:** the three modes on the plain (broken) read from Ticket 3. A repeated read is a good moment to put it in a function, as a developer would.

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
def read_reviews(mode, fixed=False):
    # fixed=True -> the correct multiLine read from Ticket 3
    options = dict(multiLine=True, quote='"', escape='"') if fixed else {}
    return spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL, mode=mode, **options)


MODES = ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]

for mode in MODES:
    print(f"{mode:<14} .count() -> {read_reviews(mode).count():,}")
```

In [17]:
def read_reviews(mode, fixed=False):
    # fixed=True -> the correct multiLine read from Ticket 3
    options = dict(multiLine=True, quote='"', escape='"') if fixed else {}
    return spark.read.csv(REVIEWS, header=True, schema=REVIEW_DDL, mode=mode, **options)


MODES = ["PERMISSIVE", "DROPMALFORMED", "FAILFAST"]

for mode in MODES:
    print(f"{mode:<14} .count() -> {read_reviews(mode).count():,}")

PERMISSIVE     .count() -> 104,162
DROPMALFORMED  .count() -> 104,162


FAILFAST       .count() -> 104,162


📝 **Observe.** All three modes gave the same number and `FAILFAST` did not fail. Were you surprised? Before you read on, write your own guess: why did Spark not notice the bad rows?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

**The reason:** `.count()` does not need any column values, so Spark **never parses the columns** and never sees the bad rows. Spark skips work it does not need (you will meet this idea again in Lab 4). To force it to really look at `review_score`, ask for something that uses it:

⌨️ **Type this.** Type this into the cell below (do not paste), then run it with Shift+Enter:

```python
def rows_and_scores(df):
    r = df.agg(F.count(F.lit(1)).alias("rows"), F.count("review_score").alias("scores")).first()
    return r.rows, r.scores


def run_modes(fixed):
    out = {}
    for mode in MODES:
        try:
            out[mode] = rows_and_scores(read_reviews(mode, fixed))
            print(f"{mode:<14} rows = {out[mode][0]:>7,}   review_score filled in = {out[mode][1]:>7,}")
        except Exception as e:
            out[mode] = None
            print(f"{mode:<14} FAILED. Root cause:\n      {root_cause(e)}")
    return out


print("--- the plain (broken) read ---")
broken = run_modes(fixed=False)

print("\n--- the correct multiLine read ---")
clean = run_modes(fixed=True)
```

In [18]:
def rows_and_scores(df):
    r = df.agg(F.count(F.lit(1)).alias("rows"), F.count("review_score").alias("scores")).first()
    return r.rows, r.scores


def run_modes(fixed):
    out = {}
    for mode in MODES:
        try:
            out[mode] = rows_and_scores(read_reviews(mode, fixed))
            print(f"{mode:<14} rows = {out[mode][0]:>7,}   review_score filled in = {out[mode][1]:>7,}")
        except Exception as e:
            out[mode] = None
            print(f"{mode:<14} FAILED. Root cause:\n      {root_cause(e)}")
    return out


print("--- the plain (broken) read ---")
broken = run_modes(fixed=False)

print("\n--- the correct multiLine read ---")
clean = run_modes(fixed=True)

--- the plain (broken) read ---


PERMISSIVE     rows = 104,162   review_score filled in =  99,225


DROPMALFORMED  rows = 101,605   review_score filled in =  99,225
FAILFAST       FAILED. Root cause:
      Caused by: org.apache.spark.SparkException: [MALFORMED_RECORD_IN_PARSING.WITHOUT_SUGGESTION] Malformed records are detected in record parsing: [null].
      Caused by: java.lang.NumberFormatException: For input string: " que prejudica a compra. O razoável para minha cidade seria uma semana máximo""

--- the correct multiLine read ---


PERMISSIVE     rows =  99,224   review_score filled in =  99,224


DROPMALFORMED  rows =  99,224   review_score filled in =  99,224
FAILFAST       rows =  99,224   review_score filled in =  99,224


📝 **Observe.** (a) What did each mode do with the bad rows now? (b) Read the `Caused by` line of the `FAILFAST` error: which value landed in which column? (c) Why do all three modes agree on the correct read? Where does the real fix belong: in the mode, or in the read options?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

✍️ **Now you.** 
Do the strict read on **orders**. Write a DDL string called **`ORDERS_DDL`** for the 8 columns (the 5 date columns are `TIMESTAMP`, the rest `STRING`; you saw the names in Ticket 1), read the file into **`orders_typed`** with `header=True`, your schema and `mode="FAILFAST"`. Then store in **`n_delivered`** how many orders have an `order_delivered_customer_date`. Use `F.count(column)` inside `agg`, so Spark really parses that column.

In [19]:
ORDERS_DDL = ("order_id STRING, customer_id STRING, order_status STRING, order_purchase_timestamp TIMESTAMP, "
              "order_approved_at TIMESTAMP, order_delivered_carrier_date TIMESTAMP, "
              "order_delivered_customer_date TIMESTAMP, order_estimated_delivery_date TIMESTAMP")
orders_typed = spark.read.csv(ORDERS, header=True, schema=ORDERS_DDL, mode="FAILFAST")
n_delivered = orders_typed.agg(F.count("order_delivered_customer_date")).first()[0]
orders_typed.printSchema()
print(n_delivered)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

96476


📝 **Observe.** Did `FAILFAST` raise an error on orders? What does that tell you about the orders file compared with the reviews file? How many orders have a delivery date, and what do you think the others are?

> **I saw:** *(double-click this cell, replace this line with your own words, press Shift+Enter)*

In [20]:
check("plain read + FAILFAST: raises an error", lambda: broken["FAILFAST"] is None)
check("correct read: all three modes agree on 99,224 reviews",
      lambda: all(v == (99_224, 99_224) for v in clean.values()), "go back to the multiLine read in Ticket 3")
check("orders_typed: 5 timestamp columns",
      lambda: sum(t == "timestamp" for _, t in orders_typed.dtypes) == 5, "which columns hold dates?")
check("n_delivered = 96,476", lambda: n_delivered == 96_476)

✅ plain read + FAILFAST: raises an error
✅ correct read: all three modes agree on 99,224 reviews
✅ orders_typed: 5 timestamp columns
✅ n_delivered = 96,476


---
## Wrap up

In [21]:
checks_summary(expected=16)


🎉 16/16 checks passed


### The moves you typed in this lab
Say each one out loud, then try to type it without looking.

| Move | What it looks like |
|---|---|
| Read a CSV with its header | `spark.read.csv(path, header=True)` |
| Declare types yourself | `spark.read.csv(path, header=True, schema=DDL)` |
| Read multi-line text fields | `..., multiLine=True, quote='"', escape='"'` |
| Choose what happens to bad rows | `..., mode="FAILFAST"` |
| The three-line habit | `df.printSchema()` · `df.show(5)` · `df.count()` |
| Filter rows | `df.filter(F.col("review_score") == 1)` |
| Count non-null values | `df.agg(F.count("column")).first()[0]` |
| Pick and rename columns | `df.select("a", F.col("b").alias("c"))` |
| Join two DataFrames | `left.join(right, "key")` |
| Chain steps over several lines | `(df.filter(...).select(...).show())` |

### Before you move on
1. **Restart the kernel and Run All.** Every cell should finish without an error.
2. **Fill the Lab 1 section of `OBSERVATION_REPORT.md`:** the row-count ledger, the errors you hit, and the moves you can type from memory.
3. Commit your work:

```bash
git add notebooks/01_read.ipynb OBSERVATION_REPORT.md
git commit -m "Lab 1: read Olist correctly (header, schema, multiLine, modes)"
```

**You can say out loud, without looking:** why `inferSchema` costs a pass over the data and an explicit schema does not, why a zip code is a string, and why a row count that looks plausible can still be wrong.

*Next: Lab 2 · Transform.*